# 08 — Ablation cho paper trên Google Colab A100

Chạy **toàn bộ** thí nghiệm của 5 bảng (`experiments/stage6_ablation.yaml`) quanh mô hình gốc **M0** = Swin-B-384 ·
MoE hướng D · d_model 256 · r5_nobce · clin + derm + metadata (test Dice 0.512), rồi xuất **một file Excel** chứa mọi bảng.

**Tận dụng A100:**
- **bf16 + TF32** (`configs/env/colab_a100.yaml`): kiểu số gốc của A100, không cần GradScaler.
- **Batch 16 × accum 4, không grad checkpointing**: 32 ảnh 384px mỗi bước, batch hiệu dụng vẫn 64 như mọi run khác.
- **Số job song song theo VRAM**: A100 80GB chạy 2 run cùng lúc trên 1 GPU, A100 40GB chạy 1 run.
- **DataLoader**: số worker theo số CPU, giữ worker giữa các epoch, prefetch 4 batch.
- Train trên ổ SSD của Colab. Một tiến trình nền **đồng bộ kết quả lên Google Drive mỗi 5 phút** (bỏ qua checkpoint).

**Cách chạy**
1. Runtime → Change runtime type → **A100 GPU**, bật **High-RAM** (cần Colab Pro hoặc Pay-as-you-go).
2. Runtime → **Run all**, cho phép truy cập Google Drive khi được hỏi.
3. Colab ngắt giữa chừng: mở lại notebook và **Run all**. Run đã xong (đã đồng bộ lên Drive) được bỏ qua, run dở dang
   được chạy lại. Gói Pro thường cần **giữ tab mở**; chỉ Pro+ chạy nền khi đóng trình duyệt.
4. Kết quả: `MyDrive/milk10k_ablation/outputs/ablation_tables/ablation_results.xlsx` (+ 5 bảng markdown, file nộp).

In [ ]:
# ===================== CẤU HÌNH =====================
REPO_URL = "https://github.com/trong5nhan6/fussion.git"
BRANCH = "main"
DRIVE_URL = "https://drive.google.com/file/d/1lM1nB_4pq3Vg-GwlaTMQjtus_bpbMhGX/view?usp=drive_link"  # zip dữ liệu MILK10k
DRIVE_DIR = "/content/drive/MyDrive/milk10k_ablation"   # kết quả lưu ở đây, dùng để chạy tiếp khi Colab ngắt

# ===================== CHỌN THÍ NGHIỆM =====================
P1 = ["M0", "CD", "E", "t0"]                                   # bắt buộc, 5 fold
P2 = ["C", "D", "CM", "DM", "A", "B", "t1", "nosamp", "aux0", "md0", "d1024", "sw224"]   # nên có, fold 0
VARIANTS = P1 + P2       # toàn bộ 32 run | chỉ P1 | danh sách tự chọn
FOLDS = None             # None = theo yaml (P1: 0–4, P2: 0)

TIME_BUDGET_H = 23.0     # không bắt đầu run mới nếu không kịp xong trước mốc này (phiên Colab tối đa ~24 giờ)
JOBS_PER_GPU = "auto"    # auto: 2 nếu VRAM >= 70GB (A100 80GB), 1 nếu 40GB | hoặc số cụ thể

# (Tuỳ chọn) Bảng 4: giải nén output notebook 07 (Kaggle) vào thư mục này trên Drive
REFERENCE = [DRIVE_DIR + "/nb07_outputs"]
# Dice test leaderboard của các file đã nộp -> điền vào bảng. Khoá = tên run hoặc tên thư mục trong submissions/
TEST_SCORES = {
    "swin_base_384__D__r5_nobce__clin+derm": 0.512,
    "vit_base__D__r5_nobce__clin+derm": 0.479,
    # "swin_base_384__abl_M0__cv5": None,
    # "swin_base_384__abl_M0__cv5_thr": None,
}

In [ ]:
# ===================== THAM SỐ TRAIN =====================
# Ghi đè mọi config (kể cả set chung của lưới stage6: batch 8 x 8 + grad checkpointing dành cho T4).
PARAMS = {
    "train.batch_size": 16,            # 16 lesion = 32 ảnh 384px; CUDA out of memory -> 8 và accum_steps 8
    "train.accum_steps": 4,            # batch hiệu dụng 16 x 4 = 64 (giống mọi run trước)
    "model.grad_checkpointing": False, # A100 đủ VRAM -> không tính lại activation (nhanh hơn ~25%)
    "train.amp_dtype": "bf16",
    "train.epochs": 20,
    "train.patience": 5,
    "seed": 42,
    "predict.tta": True,
}
RUN_SUFFIX = ""

## 1. Google Drive, GPU và cấu hình song song

In [ ]:
import os, subprocess, torch
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_DIR + "/outputs", exist_ok=True)

!nvidia-smi --query-gpu=name,memory.total --format=csv
assert torch.cuda.is_available(), "Chưa bật GPU: Runtime -> Change runtime type -> A100 GPU"
props = torch.cuda.get_device_properties(0)
vram = props.total_memory / 2**30
if "A100" not in props.name:
    print(f"CẢNH BÁO: đang dùng {props.name}, không phải A100. bf16 cần GPU Ampere trở lên (A100, L4, H100).")
jobs = JOBS_PER_GPU if JOBS_PER_GPU != "auto" else (2 if vram >= 70 else 1)
cpus = os.cpu_count()
PARAMS["data.num_workers"] = max(2, cpus // jobs - 1)
GPUS = ",".join(["0"] * jobs)
print(f"{props.name} {vram:.0f}GB | {cpus} CPU | {jobs} run song song | {PARAMS['data.num_workers']} worker/run")

## 2. Code và dữ liệu

In [ ]:
REPO_DIR = "/content/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
DATA_DIR = "/content/MILK10k"            # ổ SSD của Colab: đọc ảnh nhanh hơn Drive
os.environ["MILK10K_ROOT"] = DATA_DIR
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q "timm>=1.0" gdown openpyxl
!git log -1 --format="Commit: %h %s"
!python scripts/setup_data.py --drive-url "{DRIVE_URL}" --dest {DATA_DIR}

In [ ]:
# Kiểm tra nhanh toàn bộ pipeline (~2 phút)
!python -m pytest tests -q -p no:cacheprovider

## 3. Đồng bộ kết quả lên Drive và danh sách run

In [ ]:
# Chạy tiếp: run đã xong trên Drive được chép về outputs/ và bỏ qua
restore_arg = f"--restore {DRIVE_DIR}/outputs"
# Đồng bộ nền mỗi 5 phút (bỏ checkpoint .pt) + ghi mức dùng GPU mỗi phút vào outputs/_gpu_usage.csv
os.makedirs("outputs", exist_ok=True)
sync_cmd = f"while true; do rsync -a --exclude '*.pt' outputs/ '{DRIVE_DIR}/outputs/'; sleep 300; done"
if "sync_proc" not in globals() or sync_proc.poll() is not None:
    sync_proc = subprocess.Popen(["bash", "-c", sync_cmd])
    gpu_log = subprocess.Popen("nvidia-smi --query-gpu=timestamp,utilization.gpu,memory.used,power.draw "
                               "--format=csv -l 60 > outputs/_gpu_usage.csv", shell=True)
print("Đồng bộ nền:", DRIVE_DIR + "/outputs")

In [ ]:
import shlex, yaml

def to_set_args(params: dict) -> str:
    # Chuyển PARAMS thành chuỗi --set key=value cho scripts (bỏ qua giá trị None)
    items = []
    for k, v in params.items():
        if v is None:
            continue
        val = yaml.safe_dump(v, default_flow_style=True).strip().removesuffix("...").strip()
        items.append(shlex.quote(f"{k}={val}"))
    return ("--set " + " ".join(items)) if items else ""

set_arg = to_set_args(PARAMS)
suffix_arg = f"--suffix {RUN_SUFFIX}" if RUN_SUFFIX else ""
print("Override:", set_arg or "(không có)")
print("Hậu tố tên run:", RUN_SUFFIX or "(không có)")

In [ ]:
import yaml
exp = yaml.safe_load(open("experiments/stage6_ablation.yaml", encoding="utf-8"))
select_arg = f"--variants {','.join(VARIANTS)}" + (f" --folds {','.join(map(str, FOLDS))}" if FOLDS else "")

# Phút / run trên A100 (ước lượng từ T4 đo thật: bf16 + không grad checkpointing nhanh hơn khoảng 4 lần)
def minutes(v):
    spec = exp["variants"][v]
    if "224" in spec.get("config", ""):
        return 15
    return 20 if len(spec.get("views", ["clin", "derm"])) == 1 else 45
n_folds = lambda v: len(FOLDS) if FOLDS else len(exp["variants"][v].get("folds", [0]))
total = sum(minutes(v) * n_folds(v) for v in VARIANTS)
speed = 1.6 if jobs == 2 else 1.0     # 2 run cùng GPU: thông lượng tăng khoảng 1.6 lần (ước lượng)
print(f"{sum(n_folds(v) for v in VARIANTS)} run, ước tính ~{total / 60 / speed:.1f} giờ trên A100 "
      f"({jobs} run song song). Số đo thật in ra sau run đầu tiên.")
!python scripts/run_grid.py experiments/stage6_ablation.yaml --env configs/env/colab_a100.yaml {select_arg} {set_arg} {suffix_arg} {restore_arg} --dry-run

## 4. Train

In [ ]:
!python scripts/run_grid.py experiments/stage6_ablation.yaml --env configs/env/colab_a100.yaml {select_arg} {set_arg} {suffix_arg} {restore_arg} --gpus {GPUS} --est-run-h 0.8 --time-budget-h {TIME_BUDGET_H}

## 5. Bảng kết quả và file Excel
Gộp mọi run (và `REFERENCE`): 5 bảng, Δ so với M0 ghép cặp theo fold, ensemble test 5 fold, ngưỡng theo lớp.
Nộp file trong `outputs/ablation_tables/submissions/` lên leaderboard, điền điểm vào `TEST_SCORES` rồi **chỉ chạy lại ô này**.

In [ ]:
from IPython.display import Markdown, display
yaml.safe_dump({k: v for k, v in TEST_SCORES.items() if v is not None}, open("outputs/test_scores.yaml", "w"))
ref_dirs = " ".join(r for r in REFERENCE if os.path.isdir(r))
!python scripts/ablation_tables.py --dirs outputs {ref_dirs} --test-scores outputs/test_scores.yaml > /dev/null
display(Markdown(open("outputs/ablation_tables/ablation_tables.md", encoding="utf-8").read()))
!rsync -a --exclude '*.pt' outputs/ "{DRIVE_DIR}/outputs/"
print("Excel:", DRIVE_DIR + "/outputs/ablation_tables/ablation_results.xlsx")

In [ ]:
# Mức dùng GPU trong lúc train (lấy mẫu mỗi phút)
import pandas as pd
u = pd.read_csv("outputs/_gpu_usage.csv", skipinitialspace=True)
util = u.filter(like="utilization").iloc[:, 0].str.rstrip(" %").astype(float)
mem = u.filter(like="memory.used").iloc[:, 0].str.rstrip(" MiB").astype(float) / 1024
print(f"GPU util trung bình {util.mean():.0f}% (trung vị {util.median():.0f}%), VRAM tối đa {mem.max():.1f}GB, {len(u)} mẫu")